In [37]:
from torch import nn

VOCAB = 26*2 + 5

class TakeOutput(nn.Module):
    def forward(self, x):
        out, h = x
        return out

model = nn.Sequential(
    nn.RNN(VOCAB, VOCAB*2, 3),
    TakeOutput(),
    nn.Linear(VOCAB*2, VOCAB),
    nn.Softmax(-1)
)

model

Sequential(
  (0): RNN(57, 114, num_layers=3)
  (1): TakeOutput()
  (2): Linear(in_features=114, out_features=57, bias=True)
  (3): Softmax(dim=-1)
)

In [4]:
print(list(map(ord, ' ,.?AZaz')))

[32, 44, 46, 63, 65, 90, 97, 122]


In [5]:
print(chr(45))
for i in range(90,98):
    print(chr(i))

-
Z
[
\
]
^
_
`
a


In [7]:
122-56

66

In [16]:
from torch import zeros, Tensor, argmax

def _encode_pos(c: str) -> int:
    if c == ' ': return 0
    if c == ',': return 1
    if c == '-': return 2
    if c == '.': return 3
    if c == '?': return 4
    
    j = ord(c)
    if 65 <= j <= 90: return j - 60
    if 97 <= j <= 122: return j - 66
    
    return 4

def _encode(c: str) -> Tensor:
    out = zeros(VOCAB)
    i = _encode_pos(c)
    out[i] = 1
        
    return out

def _decode_pos(i: int) -> str:
    if i == 0: return ' '
    if i == 1: return ','
    if i == 2: return '-'
    if i == 3: return '.'
    if i == 4: return '?'
    
    if 5 <= i <= 30: return chr(i+60)
    if 31 <= i <= 56: return chr(i+66)
    
    return '?'


def _decode(t: Tensor) -> str:
    i = int(t.argmax())
    return _decode_pos(i)

In [18]:
import string

for c in string.ascii_letters + ' ,-.':
    i = _encode_pos(c)
    c_ = _decode_pos(i)
    assert c == c_, (c,i,c_)
    i = _encode(c)
    c_ = _decode(i)
    assert c == c_, (c,i,c_)

In [ ]:
from torch import stack

def encode(s: str) -> Tensor:
    return stack(tuple(map(_encode, s))) #.unsqueeze(1)

def decode(t: Tensor) -> str:
    return ''.join(map(_decode, t))

'hello'

In [33]:
inp = encode("hello")
decode(inp)

'hello'

In [38]:
out = model(inp)
decode(out)

'Be  e'